# Modelo de risco a partir de SNPs

Notebook educacional com dados **simulados**. Não serve para diagnóstico.

Etapas: simular genótipos, treinar Random Forest e avaliar com ROC AUC.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(42)

## 1. Dados simulados

Cada SNP é codificado como 0, 1 ou 2 (número de alelos alternativos). O risco depende de dois SNPs, mais ruído.

In [ ]:
n, n_snps = 1000, 20
X = pd.DataFrame(rng.integers(0, 3, size=(n, n_snps)), columns=[f"snp_{i}" for i in range(n_snps)])
logit = 0.8 * X["snp_0"] + 0.5 * X["snp_1"] - 1.2 + rng.normal(0, 0.5, n)
y = (1 / (1 + np.exp(-logit)) > rng.random(n)).astype(int)
X.head()

## 2. Treino e avaliação

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
modelo = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_train, y_train)
roc_auc_score(y_test, modelo.predict_proba(X_test)[:, 1])

## 3. Importância dos SNPs

In [ ]:
pd.Series(modelo.feature_importances_, index=X.columns).sort_values(ascending=False).head(5)